In [ ]:
# vectorbt crypto 回测框架
# 核心思路 生成 vectorbt.Portfolio.from_orders()所需要的close，price，size
# 
import pandas as pd
import numpy as np
import vectorbt
import ccxt
import ccxt.pro as ccxtpro
import warnings
import time
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
vectorbt.settings['plotting']['layout']['width'] = 1200
vectorbt.settings['plotting']['layout']['height'] = 400

In [68]:
# ccxt 参数配置
# 假设使用binance,获取行情数据相关不需要api key相关配置
# ex为普通接口，ex_pro为异步接口，可使用ws
ex = ccxt.binance({
    'apiKey': '',
    'secret': '',
    'proxies': {
        'http': 'http://127.0.0.1:7890',
        'https': 'http://127.0.0.1:7890',
    }
})
ex_pro = ccxtpro.binance({
    'apiKey': '',
    'secret': '',
})
# 设置ccxt pro代理
ex_pro.httpProxy = "http://127.0.0.1:7890"
ex_pro.wsProxy = 'http://127.0.0.1:7890'

In [55]:
class Strategy:
    # df 至少需要包含 o,h,l,c,v,     index为datetime格式
    # orders 需要包含 c，entry，exit，price，size, index为datetime格式
    def __init__(self, data=pd.DataFrame()):
        self.df = data
        self.orders = pd.DataFrame()
    # 以双均线为教程
    def signal(self):
        entrys,exits,pos = {},{},{}
        last_pos = 0.0
        res_df = pd.DataFrame()
        res_df['c'] = self.df['c']
        #指标计算
        ma_short = self.df['c'].rolling(10).mean()
        ma_long = self.df['c'].rolling(20).mean()
        for i, (index, row) in enumerate(self.df.iterrows()):
            entrys[index] = False
            exits[index] = False
            # 20天后才有双均线数据，前20天不交易
            if i>20:
                # 短线从下至上穿过长线 买入
                if ma_short[index]>=ma_long[index] and ma_short.shift(1)[index]<ma_long.shift(1)[index]:
                    entrys[index] = True
                    last_pos += 1.0
                # 短线从上至下穿过长线 卖出
                if ma_short[index]<=ma_long[index] and ma_short.shift(1)[index]>ma_long.shift(1)[index]:
                    exits[index] = True
                    last_pos -= 1.0
            pos[index] = last_pos
        # for 结束
        res_df['entry'] = entrys.values()
        res_df['exit'] = exits.values()
        res_df['pos'] = pos.values()
        res_df['price'] = res_df['c']
        # 计算size
        res_df['size'] = res_df['pos']-res_df['pos'].shift(1)
        res_df['size'].fillna(0,inplace=True)
        self.orders = res_df
        return self
    # vectorbt回测
    # size_factor可以根据实际进行调整
    def backtest(self, init_cash=10000., fees=0.001,size_factor=1):
        pf = vectorbt.Portfolio.from_orders(
            self.orders['c'],
            size=self.orders['size']*size_factor,
            init_cash=init_cash,
            price=self.orders['price'],
            fees=fees,
            freq='1d'
        )
        return pf

In [56]:
# exchange为ex类型，ex_pro需要异步
# 获取最近几千条k线数据
def get_klines(exchange, symbol, count=1):
    df = pd.DataFrame(exchange.fetch_ohlcv(symbol, timeframe='1m', limit=1000))
    for i in range(count - 1):
        since = df[0][0] - 1000 * 60 * 1000
        res_df = pd.DataFrame(exchange.fetch_ohlcv(symbol, timeframe='1m', since=since, limit=1000))
        df = pd.concat([res_df, df], ignore_index=True)
        time.sleep(0.2)
    df.columns = ['t', 'o', 'h', 'l', 'c', 'v']
    df['datetime'] = pd.to_datetime(df['t'], unit='ms', origin='1970-01-01 08:00:00')
    df.set_index('datetime', inplace=True)
    return df
# 检查数据连续性
def check_data(data: pd.DataFrame):
    duplicated_ok = data[data.index.duplicated()].empty
    diff = data['t'].diff()[1:]
    diff_ok = len(diff[diff == diff.iloc[0]]) == len(diff)
    if not duplicated_ok:
        print('数据有重复')
    if not diff_ok:
        print('数据不连续')
    return duplicated_ok and diff_ok


In [57]:
data = get_klines(ex,"BTCUSDT",1)
check_data(data)

True

In [58]:
data

,t,o,h,l,c,v
datetime,,,,,,
2024-11-23 23:26:00,1732375560000,98696.96,98726.38,98688.01,98726.38,22.87001
2024-11-23 23:27:00,1732375620000,98726.38,98750.77,98724.00,98724.01,53.58766
2024-11-23 23:28:00,1732375680000,98724.00,98724.70,98668.26,98668.26,35.20324
2024-11-23 23:29:00,1732375740000,98668.27,98670.00,98668.26,98669.99,6.34318
2024-11-23 23:30:00,1732375800000,98670.00,98670.00,98639.61,98639.61,10.18561
...,...,...,...,...,...,...
2024-11-24 16:01:00,1732435260000,98449.24,98449.25,98381.10,98422.27,25.68902
2024-11-24 16:02:00,1732435320000,98422.26,98427.27,98422.26,98427.26,29.03765
2024-11-24 16:03:00,1732435380000,98427.27,98427.29,98417.24,98417.24,15.59961


In [59]:
# 订单
Strategy(data).signal().orders

,c,entry,exit,pos,price,size
datetime,,,,,,
2024-11-23 23:26:00,98726.38,False,False,0.0,98726.38,0.0
2024-11-23 23:27:00,98724.01,False,False,0.0,98724.01,0.0
2024-11-23 23:28:00,98668.26,False,False,0.0,98668.26,0.0
2024-11-23 23:29:00,98669.99,False,False,0.0,98669.99,0.0
2024-11-23 23:30:00,98639.61,False,False,0.0,98639.61,0.0
...,...,...,...,...,...,...
2024-11-24 16:01:00,98422.27,False,False,1.0,98422.27,0.0
2024-11-24 16:02:00,98427.26,False,False,1.0,98427.26,0.0
2024-11-24 16:03:00,98417.24,False,False,1.0,98417.24,0.0


In [60]:
# 回测
pf = Strategy(data).signal().backtest(init_cash=10000., fees=0.001, size_factor=1)
pf.stats()

Start                                2024-11-23 23:26:00
End                                  2024-11-24 16:05:00
Period                                1000 days 00:00:00
Start Value                                      10000.0
End Value                                    3413.307303
Total Return [%]                              -65.866927
Benchmark Return [%]                           -0.305734
Max Gross Exposure [%]                             100.0
Total Fees Paid                              5489.893467
Max Drawdown [%]                               65.869715
Max Drawdown Duration                  911 days 00:00:00
Total Trades                                          57
Total Closed Trades                                   56
Total Open Trades                                      1
Open Trade PnL                                 -3.639296
Win Rate [%]                                    7.142857
Best Trade [%]                                  0.093455
Worst Trade [%]                

In [64]:
# 打印回测图表
# pf.plot()

In [77]:
# ccxt pro websocket获取实时数据例子
async def get_ws_data():
    arr = []
    while True:
        res = await ex_pro.watch_ohlcv('BTCUSDT', timeframe='1s', limit=1);
        df = pd.DataFrame(res)
        df.columns = ["t", "o", "h", "l", "c", "v"]
        df['datetime'] = pd.to_datetime(df['t'], unit='ms', origin='1970-01-01 08:00:00')
        df.set_index('datetime', inplace=True)
        print(df)
        # arr.append(res)
await get_ws_data()

                                 t         o         h         l         c        v
datetime                                                                           
2024-11-24 16:15:55  1732436155000  98424.24  98424.24  98421.32  98421.32  0.18864
                                 t         o         h         l         c        v
datetime                                                                           
2024-11-24 16:15:56  1732436156000  98421.32  98421.33  98421.32  98421.33  2.30246
                                 t         o         h         l         c        v
datetime                                                                           
2024-11-24 16:15:57  1732436157000  98421.33  98421.33  98421.32  98421.32  0.00446
                                 t         o         h         l         c        v
datetime                                                                           
2024-11-24 16:15:58  1732436158000  98421.32  98421.33  98421.32  98421.33  

CancelledError: 